## Lab 8 - Genetic Algorithm


In [ ]:
import random

POPULATION_SIZE = 100

GENES = '''abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ 1234567890, .-;:_!"#%&/()=?@${[]}'''

TARGET = "I study at FAST"

class Individual(object):
    def __init__(self, chromosome):
        self.chromosome = chromosome
        self.fitness = self.cal_fitness()

    @classmethod
    def mutated_genes(self):
        global GENES
        return random.choice(GENES)

    @classmethod
    def create_gnome(self):
        global TARGET
        return [self.mutated_genes() for _ in range(len(TARGET))]

    def mate(self, par2):
        child_chromosome = []
        for gp1, gp2 in zip(self.chromosome, par2.chromosome):
            prob = random.random()
            if prob < 0.45:
                child_chromosome.append(gp1)
            elif prob < 0.90:
                child_chromosome.append(gp2)
            else:
                child_chromosome.append(self.mutated_genes())
        return Individual(child_chromosome)

    def cal_fitness(self):
        global TARGET
        return sum(1 for gs, gt in zip(self.chromosome, TARGET) if gs != gt)


def main():
    global POPULATION_SIZE
    generation = 1
    found = False
    population = [Individual(Individual.create_gnome()) for _ in range(POPULATION_SIZE)]

    while not found:
        population = sorted(population, key=lambda x: x.fitness)

        if population[0].fitness <= 0:
            found = True
            break

        new_generation = []
        s = int((10 * POPULATION_SIZE) / 100)
        new_generation.extend(population[:s])

        s = int((90 * POPULATION_SIZE) / 100)
        for _ in range(s):
            parent1 = random.choice(population[:50])
            parent2 = random.choice(population[:50])
            child = parent1.mate(parent2)
            new_generation.append(child)

        population = new_generation
        print(f"Generation: {generation}\tString: {''.join(population[0].chromosome)}\tFitness: {population[0].fitness}")
        generation += 1

    print(f"Generation: {generation}\tString: {''.join(population[0].chromosome)}\tFitness: {population[0].fitness}")


main()

# Lab Task — The Knapsack Problem

**Items (from lab manual):**

| Item | Value | Weight (kg) |
|------|-------|-------------|
| N1   | 14    | 1           |
| N2   | 23    | 3           |
| N3   | 8     | 7           |
| N4   | 9     | 4           |
| N5   | 17    | 5           |
| N6   | 15    | 6           |

**Capacity:** 10 kg  
**Selection:** Tournament Selection (k=3)  
**Crossover:** One-point crossover  


In [ ]:
import random

values  = [14, 23, 8, 9, 17, 15]
weights = [1,  3,  7, 4,  5,  6]
max_weight     = 10
population_size = 10
num_generations = 50
mutation_rate   = 0.1
tournament_k    = 3

In [ ]:
def fitness(individual):
    total_value  = 0
    total_weight = 0
    for i, gene in enumerate(individual):
        if gene == 1:
            total_value  += values[i]
            total_weight += weights[i]
    if total_weight > max_weight:
        return 0
    return total_value

In [ ]:
def selection(population):
    tournament = random.sample(population, tournament_k)
    return max(tournament, key=fitness)

In [ ]:
def crossover(parent1, parent2):
    crossover_point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:crossover_point] + parent2[crossover_point:]
    child2 = parent2[:crossover_point] + parent1[crossover_point:]
    return child1, child2

In [ ]:
def mutation(individual):
    return [1 - gene if random.random() < mutation_rate else gene
            for gene in individual]

In [ ]:
population = [[random.randint(0, 1) for _ in range(len(weights))]
              for _ in range(population_size)]

for generation in range(num_generations):
    new_population = []

    while len(new_population) < population_size:
        parent1 = selection(population)
        parent2 = selection(population)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutation(child1)
        child2 = mutation(child2)
        new_population.extend([child1, child2])

    new_population = new_population[:population_size]

    combined = population + new_population
    combined.sort(key=fitness, reverse=True)
    population = combined[:population_size]

    best = max(population, key=fitness)
    print(f"Generation {generation + 1:2d} | Best chromosome: {best} | Fitness (Value): {fitness(best)}")

best_solution = max(population, key=fitness)
selected_items = [f"N{i+1}" for i, gene in enumerate(best_solution) if gene == 1]
total_w = sum(weights[i] for i, g in enumerate(best_solution) if g == 1)

print("\n=== Final Result ===")
print(f"Best chromosome : {best_solution}")
print(f"Selected items  : {selected_items}")
print(f"Total value     : {fitness(best_solution)}")
print(f"Total weight    : {total_w} kg")

# Practice Question (Take Home) — Secret Word Guessing via Genetic Algorithm

- Population size: 50  
- Each individual: string of 6 random characters  
- Fitness: number of characters matching the secret word **in the correct position**  
- Selection: fitness-proportionate  
- Crossover: one-point  
- Mutation: random character replacement  
- Elitism: best individuals carried forward  


In [ ]:
import random
import string

SECRET_WORD   = "python"
WORD_LENGTH   = len(SECRET_WORD)
POP_SIZE      = 50
MUTATION_RATE = 0.1
GENE_POOL     = string.ascii_lowercase


def create_individual():
    return [random.choice(GENE_POOL) for _ in range(WORD_LENGTH)]


def word_fitness(individual):
    return sum(1 for g, s in zip(individual, SECRET_WORD) if g == s)


def word_selection(population):
    fitness_values = [word_fitness(ind) for ind in population]
    total = sum(fitness_values)
    if total == 0:
        return random.choice(population)
    probs = [f / total for f in fitness_values]
    return random.choices(population, weights=probs, k=1)[0]


def word_crossover(parent1, parent2):
    point = random.randint(1, WORD_LENGTH - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2


def word_mutation(individual):
    return [random.choice(GENE_POOL) if random.random() < MUTATION_RATE else g
            for g in individual]


population = [create_individual() for _ in range(POP_SIZE)]

generation = 1
while True:
    population.sort(key=word_fitness, reverse=True)
    best = population[0]
    best_fit = word_fitness(best)

    print(f"Generation {generation:3d} | Best: {''.join(best)} | Matches: {best_fit}/{WORD_LENGTH}")

    if best_fit == WORD_LENGTH:
        print(f"\nSecret word '{SECRET_WORD}' found in generation {generation}!")
        break

    elites = population[:5]
    new_population = list(elites)

    while len(new_population) < POP_SIZE:
        parent1 = word_selection(population)
        parent2 = word_selection(population)
        child1, child2 = word_crossover(parent1, parent2)
        child1 = word_mutation(child1)
        child2 = word_mutation(child2)
        new_population.extend([child1, child2])

    population = new_population[:POP_SIZE]
    generation += 1